# Ajnyana — Data Pipeline

Kumpulkan, bersihkan, dan merge semua sumber teks Bahasa Sunda.

**Sources:**
- Wikipedia Sunda (`su`) — ~62K artikel
- CC-100 Sundanese (`su`) — ~10M tokens
- OSCAR Sundanese — TBD

**Output:** `data/train.txt`, `data/val.txt`, `data/stats.json`

In [ ]:
from datasets import load_dataset
import re
import json
import os
import unicodedata
from collections import Counter

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Libraries OK')

## 1. Cleaning Functions

In [ ]:
def clean_text(text: str) -> str:
    if not text or not isinstance(text, str):
        return ''

    # Normalize unicode
    text = unicodedata.normalize('NFC', text)

    # Hapus HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Hapus karakter kontrol
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    return text


def is_valid(text: str, min_words: int = 20) -> bool:
    if not text:
        return False
    words = text.split()
    if len(words) < min_words:
        return False
    # Minimal 50% karakter alfabet (filter noise/kode)
    alpha = sum(c.isalpha() for c in text)
    if alpha / max(len(text), 1) < 0.5:
        return False
    return True


print('Cleaning functions OK')

## 2. Source 1 — Wikipedia Sunda

In [ ]:
print('Loading Wikipedia Sunda...')
wiki = load_dataset('wikipedia', '20220301.su', trust_remote_code=True, split='train')
print(f'Raw: {len(wiki):,} artikel')

wiki_texts = []
for item in wiki:
    text = clean_text(item['text'])
    if is_valid(text, min_words=30):
        wiki_texts.append(text)

print(f'After cleaning: {len(wiki_texts):,} artikel ({len(wiki_texts)/len(wiki)*100:.1f}% kept)')
print(f'Total words: {sum(len(t.split()) for t in wiki_texts):,}')
print(f'Sample:\n{wiki_texts[0][:300]}\n...')

## 3. Source 2 — CC-100 Sunda

In [ ]:
print('Loading CC-100 Sunda...')
cc100 = load_dataset('statmt/cc100', lang='su', split='train', trust_remote_code=True)
print(f'Raw: {len(cc100):,} dokumen')

cc_texts = []
for item in cc100:
    text = clean_text(item['text'])
    if is_valid(text, min_words=20):
        cc_texts.append(text)

print(f'After cleaning: {len(cc_texts):,} dokumen ({len(cc_texts)/len(cc100)*100:.1f}% kept)')
print(f'Total words: {sum(len(t.split()) for t in cc_texts):,}')
print(f'Sample:\n{cc_texts[0][:300]}\n...')

## 4. Source 3 — OSCAR Sunda

In [ ]:
print('Loading OSCAR Sunda...')
try:
    oscar = load_dataset('oscar-corpus/OSCAR-2301', language='su', split='train', trust_remote_code=True)
    print(f'Raw: {len(oscar):,} dokumen')

    oscar_texts = []
    for item in oscar:
        text = clean_text(item.get('text', item.get('content', '')))
        if is_valid(text, min_words=20):
            oscar_texts.append(text)

    print(f'After cleaning: {len(oscar_texts):,} dokumen ({len(oscar_texts)/len(oscar)*100:.1f}% kept)')
    print(f'Total words: {sum(len(t.split()) for t in oscar_texts):,}')
except Exception as e:
    print(f'OSCAR tidak tersedia atau error: {e}')
    oscar_texts = []

## 5. Merge + Deduplicate

In [ ]:
all_texts = wiki_texts + cc_texts + oscar_texts
print(f'Total sebelum dedup: {len(all_texts):,} dokumen')

# Exact dedup by first 100 chars
seen = set()
deduped = []
for text in all_texts:
    key = text[:100].strip().lower()
    if key not in seen:
        seen.add(key)
        deduped.append(text)

print(f'After dedup: {len(deduped):,} dokumen ({len(deduped)/len(all_texts)*100:.1f}% kept)')

total_words = sum(len(t.split()) for t in deduped)
total_chars = sum(len(t) for t in deduped)
print(f'Total words: {total_words:,}')
print(f'Total chars: {total_chars:,}')
print(f'Est. tokens (~4 chars/token): {total_chars//4:,}')

## 6. Train/Val Split + Save

In [ ]:
import random
random.seed(42)
random.shuffle(deduped)

split = int(len(deduped) * 0.98)
train_docs = deduped[:split]
val_docs   = deduped[split:]

print(f'Train: {len(train_docs):,} docs')
print(f'Val:   {len(val_docs):,} docs')

# Save
with open('../data/processed/train.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(train_docs))

with open('../data/processed/val.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(val_docs))

# Stats
stats = {
    'total_docs': len(deduped),
    'train_docs': len(train_docs),
    'val_docs':   len(val_docs),
    'sources': {
        'wikipedia': len(wiki_texts),
        'cc100':     len(cc_texts),
        'oscar':     len(oscar_texts),
    },
    'total_words': total_words,
    'est_tokens':  total_chars // 4,
}

with open('../data/processed/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('\nSaved!')
print(json.dumps(stats, indent=2))

## 7. Quality Check

In [ ]:
# Distribusi panjang dokumen
lengths = [len(t.split()) for t in deduped]
print(f'Doc length (words):')
print(f'  Min:    {min(lengths)}')
print(f'  Max:    {max(lengths)}')
print(f'  Median: {sorted(lengths)[len(lengths)//2]}')
print(f'  Mean:   {sum(lengths)//len(lengths)}')

# Sample 3 random docs
print('\n=== SAMPLE DOCS ===')
for i, doc in enumerate(random.sample(deduped, 3)):
    print(f'\n[{i+1}] ({len(doc.split())} words)')
    print(doc[:400])
    print('...')

## Next

Setelah pipeline selesai → `02_tokenizer.ipynb` — train BPE tokenizer dari corpus ini.